# HeadSwap production pipeline

Single production path only — no A/B arms, promotion checks, or experimental modes.

Routing when `ENABLE_LIGHTING_ROUTE=True`:
- single person → existing `crop_stitch`
- multi-person, normal lighting → production `crop_stitch` (r64, ref_boost 3.5)
- multi-person, dark lighting → `full_frame` (full v1.2, ref_boost 4)

Use **Runtime → Run all** on an A100 GPU. To swap a specific person in a group photo,
run the head-count cell, then set `TARGET_HEAD` to that number and re-run from there.

In [ ]:
#@title 1 · Production settings
SEED = 46
STEPS = 8
CFG = 1.0
OUTPUT_LONG_SIDE = 1024
DEBUG = False

# Which head to swap in a group photo.
#   0 = auto, use BODY_FACE_POLICY
#   1, 2, 3, 4 … = the numbered head shown by the head-count cell
TARGET_HEAD = 0
BODY_FACE_POLICY = "largest"  # only used when TARGET_HEAD = 0

# This is the production router added on this branch.
ENABLE_LIGHTING_ROUTE = True
DARK_LIGHTING_THRESHOLD = 70.0  # mean HSV-V [0,255]

# current = local geometry detector; magic_hour = MH audit + current geometry
FACE_DETECTION_BACKEND = "current"
MAGIC_HOUR_API_KEY = ""  # optional; prefer Colab Secret MAGIC_HOUR_API_KEY

REPO_BRANCH = "feature/lighting-route-and-magichour"
PINNED_COMMIT = None
USE_DRIVE = True

print(f"branch={REPO_BRANCH}")
print(f"lighting route={ENABLE_LIGHTING_ROUTE}, threshold={DARK_LIGHTING_THRESHOLD}")
print(f"face detection backend={FACE_DETECTION_BACKEND}")
print(f"target head={TARGET_HEAD or 'auto (' + BODY_FACE_POLICY + ')'}")

In [ ]:
#@title 2 · Setup repo, ComfyUI, nodes, and production models
from pathlib import Path
import os
import subprocess

assert Path("/content").exists(), "Open this notebook in Google Colab."

import torch
if not torch.cuda.is_available():
    raise SystemExit("Runtime → Change runtime type → GPU (A100 preferred).")
print("GPU:", torch.cuda.get_device_name(0))

DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_OK = Path("/content/drive/MyDrive").exists()
    except Exception as exc:
        print(f"Drive unavailable ({exc}); using /content model cache.")

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
    check=True,
)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
if PINNED_COMMIT:
    subprocess.run(["git", "-C", str(REPO), "checkout", PINNED_COMMIT], check=True)
os.chdir(REPO)

subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

import importlib.util
spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=DRIVE_OK))

subprocess.run(["bash", "scripts/setup_colab.sh"], check=True)
subprocess.run(["bash", "scripts/setup_krea2_nodes.sh"], check=True)

# Includes r64 (normal/single crop_stitch) and full v1.2 (dark multi full_frame).
subprocess.run(
    [
        "python", "scripts/download_krea2.py",
        "--comfy", os.environ["COMFYUI_PATH"],
        "--store-dir", os.environ["HEADSWAP_MODEL_STORE"],
        "--staging-dir", os.environ["HEADSWAP_STAGING_DIR"],
        "--backend", "auto", "--disable-xet", "--include-optional",
    ],
    check=True,
)

if MAGIC_HOUR_API_KEY:
    os.environ["MAGIC_HOUR_API_KEY"] = MAGIC_HOUR_API_KEY
elif FACE_DETECTION_BACKEND == "magic_hour":
    try:
        from google.colab import userdata
        os.environ["MAGIC_HOUR_API_KEY"] = userdata.get("MAGIC_HOUR_API_KEY")
    except Exception as exc:
        raise SystemExit("Add MAGIC_HOUR_API_KEY to Colab Secrets or settings cell.") from exc

print("Ready:", subprocess.getoutput("git rev-parse --short HEAD"))

In [ ]:
#@title 3 · Upload body photo, then donor face
from pathlib import Path
from google.colab import files
from IPython.display import display
from PIL import Image

REPO = Path("/content/headswap_V2")
custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
BODY_PATH = custom / "body.png"
FACE_PATH = custom / "face.png"

def upload_one(label, destination):
    print(f"Upload {label}:")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit(f"No {label} uploaded.")
    destination.write_bytes(next(iter(uploaded.values())))
    image = Image.open(destination).convert("RGB")
    image.save(destination)
    print(label, image.size)
    display(image)

upload_one("BODY / target scene", BODY_PATH)
upload_one("FACE / identity donor", FACE_PATH)

In [ ]:
#@title 4 · Count heads and pick which one to swap
import sys
from pathlib import Path

from IPython.display import display
from PIL import Image

REPO = Path("/content/headswap_V2")
sys.path.insert(0, str(REPO / "src"))

from headswap.config import load_config
from headswap.preprocess import detect_faces, pil_to_rgb_np, resize_max_keep_ar

BODY_PATH = REPO / "data" / "custom" / "body.png"
CACHE = REPO / ".cache" / "headswap_v2"
CACHE.mkdir(parents=True, exist_ok=True)

_cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
body = Image.open(BODY_PATH).convert("RGB")

# Same resize + detector ordering the pipeline uses, so the numbers below
# match body_face_index (largest face first).
probe = resize_max_keep_ar(body, int(OUTPUT_LONG_SIDE), div_by=int(_cfg.get("div_by", 16)))
HEADS = detect_faces(pil_to_rgb_np(probe), CACHE, allow_prior=False)
print(f"heads detected (current detector): {len(HEADS)}")

import importlib.util
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)
if HEADS:
    display(colab_demo.annotate_faces(probe, HEADS))
for i, h in enumerate(HEADS, start=1):
    print(f"  head {i}: box=({h.x0},{h.y0},{h.x1},{h.y1}) conf={h.conf:.2f}")

# Magic Hour head count (audit only — returns face crops, no boxes/landmarks).
MH_RESULT = None
if str(FACE_DETECTION_BACKEND).lower() in ("magic_hour", "magichour", "mh"):
    from headswap.magichour import MagicHourFaceDetectionClient
    try:
        _mh = MagicHourFaceDetectionClient()
        MH_RESULT = _mh.detect_local_image(
            BODY_PATH,
            confidence_score=float(_cfg.get("magic_hour_confidence_score", 0.5)),
            max_wait_s=float(_cfg.get("magic_hour_max_wait_s", 120.0)),
        )
        print(f"heads detected (Magic Hour): {MH_RESULT.face_count} [status={MH_RESULT.status}]")
        for i, f in enumerate(MH_RESULT.faces, start=1):
            print(f"  MH head {i}: {f.url}")
    except Exception as exc:
        print(f"Magic Hour detection failed ({type(exc).__name__}: {exc})")
        print("Check MAGIC_HOUR_API_KEY; the swap still runs on the current detector.")

if int(TARGET_HEAD) > len(HEADS):
    raise SystemExit(f"TARGET_HEAD={TARGET_HEAD} but only {len(HEADS)} head(s) detected.")
if int(TARGET_HEAD) >= 1:
    print(f"\nswapping head {TARGET_HEAD} of {len(HEADS)}")
else:
    print(f"\nswapping automatically via policy={BODY_FACE_POLICY}")
print("Change TARGET_HEAD in the settings cell and re-run this cell to pick another head.")

In [ ]:
#@title 5 · Run production pipeline
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import display
from PIL import Image

REPO = Path("/content/headswap_V2")
sys.path.insert(0, str(REPO / "src"))

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

BODY_PATH = REPO / "data" / "custom" / "body.png"
FACE_PATH = REPO / "data" / "custom" / "face.png"
assert BODY_PATH.is_file() and FACE_PATH.is_file(), "Run upload cell first."

if int(TARGET_HEAD) >= 1:
    face_policy, face_index = "index", int(TARGET_HEAD) - 1
else:
    face_policy, face_index = str(BODY_FACE_POLICY), 0

cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({
    "seed": int(SEED),
    "steps": int(STEPS),
    "cfg": float(CFG),
    "max_body_dim": int(OUTPUT_LONG_SIDE),
    "max_dim": int(OUTPUT_LONG_SIDE),
    "save_debug": bool(DEBUG),
    "verbose": bool(DEBUG),
    "enable_lighting_route": bool(ENABLE_LIGHTING_ROUTE),
    "dark_lighting_threshold": float(DARK_LIGHTING_THRESHOLD),
    "face_detection_backend": str(FACE_DETECTION_BACKEND),
    "body_face_policy": face_policy,
    "body_face_index": face_index,
    "enable_multi_face_features": False,
    "face_swap_mode": "single",
    "mask_crop_stitch": True,
})

body = Image.open(BODY_PATH).convert("RGB")
face = Image.open(FACE_PATH).convert("RGB")
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path("/content/headswap_outputs") / f"prod_{run_stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

if "PROD_PIPELINE" not in globals() or PROD_PIPELINE is None:
    runtime = get_shared_krea2_runtime(init_custom_nodes=True)
    PROD_PIPELINE = create_pipeline(cfg, runtime=runtime)
else:
    PROD_PIPELINE.cfg.update(cfg)

result = PROD_PIPELINE.run(body, face, out_dir=RUN_DIR / "debug")
RESULT_PATH = RUN_DIR / "result.png"
META_PATH = RUN_DIR / "meta.json"
result.image.save(RESULT_PATH)
META_PATH.write_text(json.dumps(result.meta, indent=2, default=str), encoding="utf-8")

route = result.meta.get("lighting_route") or {}
mh = result.meta.get("magic_hour_face_detection") or {}
print("\nPRODUCTION RESULT")
print("route:", route.get("route"))
print("reason:", route.get("reason"))
print("heads:", route.get("faces_detected", result.meta.get("body_face_count")))
print("swapped head:", TARGET_HEAD if int(TARGET_HEAD) >= 1 else f"auto ({face_policy})")
if mh:
    print("magic hour heads:", mh.get("face_count", mh.get("error")))
print("lighting metric:", route.get("lighting_metric"))
print("threshold:", route.get("dark_lighting_threshold"))
print("effective edit mode:", result.meta.get("edit_mode"))
print("LoRA:", result.meta.get("loras_loaded"))
print("ref_boost:", result.meta.get("ref_boost"))
print("latency_s:", round(result.latency_s, 2))
print("saved:", RUN_DIR)
display(result.image)

In [ ]:
#@title 6 · Download result
from google.colab import files

files.download(str(RESULT_PATH))